In [16]:
# https://apxml.com/courses/model-interpretability-explainability/chapter-2-lime-local-interpretability/lime-python-implementation

import lime
import lime.lime_tabular
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


iris = datasets.load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, random_state=0)

# Обучение логистической регрессии
logistic_model = LogisticRegression(solver='lbfgs')
logistic_model.fit(X_train, y_train)

# Creating a LIME object for tabular data
explainer = lime.lime_tabular.LimeTabularExplainer(X_train, feature_names=iris.feature_names, class_names=iris.target_names, discretize_continuous=True)

# Selecting an example from the test set for interpretation
i = 25 
exp = explainer.explain_instance(X_test[i], logistic_model.predict_proba, num_features=2)

print(exp.as_list())


result = logistic_model.predict([X_test[i]])

print(result)

print(f"Predict class name: {iris.target_names[result[0]]}")

print(f"True class name: {iris.target_names[y_test[i]]}")

print(f"---"*20)

for feature_name, value in zip(iris.feature_names, X_test[i]):
    print(f"{feature_name}: {value}")




print(f"---"*20)


proba = logistic_model.predict_proba([X_test[i]])[0]

for class_name, prob in zip(iris.target_names, proba):
    print(f"{class_name}: {prob:.4f}")

print(f"---"*20)


print(f"\n LIME EXPLANATIONS FOR EACH CLASS:")
print("="*20)

for class_idx, class_name in enumerate(iris.target_names):
    # Create a separate explanation for each class
    exp_class = explainer.explain_instance(
        X_test[i], 
        logistic_model.predict_proba, 
        num_features=4,
        labels=[class_idx]  # GET explanation FOR specific class
    )
    
    print(f"\n{class_name} (вероятность: {proba[class_idx]:.4f}):")
    try:
        features = exp_class.as_list(label=class_idx)   # GET explanation FOR specific class
        if features:
            for feature, weight in features:
                arrow = "➕" if weight > 0 else "➖"
                print(f"  {arrow} {feature}: {weight:+.4f}")
        else:
            print("  (нет значимых признаков для этого класса)")
    except KeyError:
        print("  (нет объяснения для этого класса)")






[('petal length (cm) <= 1.58', -0.41457052551020984), ('petal width (cm) <= 0.30', 0.1981600599528148)]
[0]
Predict class name: setosa
True class name: setosa
------------------------------------------------------------
sepal length (cm): 4.6
sepal width (cm): 3.6
petal length (cm): 1.0
petal width (cm): 0.2
------------------------------------------------------------
setosa: 0.9942
versicolor: 0.0058
virginica: 0.0000
------------------------------------------------------------

 LIME EXPLANATIONS FOR EACH CLASS:

setosa (вероятность: 0.9942):
  ➕ petal length (cm) <= 1.58: +0.7608
  ➕ sepal length (cm) <= 5.10: +0.0872
  ➕ sepal width (cm) > 3.30: +0.0423
  ➕ petal width (cm) <= 0.30: +0.0322

versicolor (вероятность: 0.0058):
  ➖ petal length (cm) <= 1.58: -0.3987
  ➕ petal width (cm) <= 0.30: +0.1944
  ➖ sepal length (cm) <= 5.10: -0.1323
  ➖ sepal width (cm) > 3.30: -0.0576

virginica (вероятность: 0.0000):
  ➖ petal length (cm) <= 1.58: -0.3596
  ➖ petal width (cm) <= 0.30: -0.21